<a href="https://colab.research.google.com/github/OJB-Quantum/Qiskit-Metal-to-Litho/blob/main/Layout_to_3D_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

# Layout to 3D with CuPy

The cell imports GDSII, OASIS, or DXF, reduces polygon boundary detail, and extrudes filled material from Z = 0 to the specified thickness.

| Setting | Default | Meaning |
| --- | --- | --- |
| `THICKNESS_UM` | `1.0` | Extrusion thickness in micrometres. |
| `XY_TOLERANCE_UM` | `0.1` | Boundary approximation tolerance in micrometres; 0.1 micrometre equals 100 nanometres. Set 0 to remove only redundant collinear points. |
| `USE_GPU` | `True` | Enable CuPy; set False for a CPU comparison. |
| `OUTPUT_FORMAT` | `"stl"` | Choose binary STL or Wavefront OBJ. |
| `DXF_UNIT_UM` | `1.0` | Micrometres per drawing unit for unitless DXF; declared units override this value. |

Boundary detail directly determines the number of triangles in a straight extrusion, because each remaining edge produces two wall triangles and contributes to the cap triangulation. For simple polygon boundary rings without added interior vertices, the closed mesh holds the following number of triangles:

$$
T = 4(N + H - C).
$$

Here, N counts all planar boundary vertices without duplicate closing coordinates, H counts holes, and C counts disconnected polygon regions. Triangle already avoids refinement vertices in this workflow, so reducing N before extrusion directly reduces both the triangle count and the binary STL size. Keeping hundreds of thousands of holes still requires millions of triangles, even after curved boundaries use fewer points.

| Quantity | Detailed mesh | 0.1 micrometre boundary tolerance |
| --- | ---: | ---: |
| Triangles | 25,924,628 | 9,286,968 |
| Binary STL size, decimal MB | 1,296.2 | 464.3 |
| Reduction in triangles and STL payload | Reference | 64.2% |

[Shapely simplification](https://shapely.readthedocs.io/en/stable/reference/shapely.simplify.html) · [Shapely array reconstruction](https://shapely.readthedocs.io/en/stable/reference/shapely.from_ragged_array.html) · [Triangle](https://rufat.be/triangle/API.html) · [CuPy](https://docs.cupy.dev/en/stable/user_guide/performance.html) · [Polygon triangulation count](https://www.science.smith.edu/~jorourke/books/ArtGalleryTheorems/Art_Gallery_Chapter_5.pdf)


In [3]:
!uv pip uninstall -q cupy-cuda13x cupy-cuda12x cupy
!uv pip install -q gdstk ezdxf "shapely>=2.2,<3" triangle trimesh "cupy-cuda12x[ctk]>=14,<15"

In [4]:
"""This script extrudes layouts with CPU geometry and optional CuPy mesh work."""

# --- Control Knobs ---
THICKNESS_UM = 1.0
XY_TOLERANCE_UM = 0.1  # Boundary simplification: 0.1 um = 100 nm; 0 disables it.
USE_GPU = True  # Set False for a CPU timing comparison.
OUTPUT_FORMAT = "stl"  # "stl" or "obj".
DXF_UNIT_UM = 1.0  # Used only when the DXF declares no units.

# --- Import ---
import struct
from pathlib import Path
from time import perf_counter

import ezdxf
import gdstk
import numpy as np
import shapely
import triangle
import trimesh
from ezdxf import units
from ezdxf.addons import geo
from ezdxf.disassemble import recursive_decompose
from google.colab import files
from IPython.display import display
from shapely.affinity import scale
from shapely.geometry import shape
from trimesh.viewer.notebook import scene_to_notebook

# --- Detect and Initialize CUDA ---
setup_started = perf_counter()
gpu_available = False
if USE_GPU:
    try:
        import cupy as cp

        if cp.cuda.runtime.getDeviceCount() > 0:
            (cp.arange(16, dtype=cp.float64) * 2).sum().get()
            cp.cuda.get_current_stream().synchronize()
            gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"]
            if isinstance(gpu_name, bytes):
                gpu_name = gpu_name.decode()
            gpu_available = True
            print(f"GPU enabled: {gpu_name}", flush=True)
        else:
            print("No CUDA GPU detected; using CPU.", flush=True)
    except Exception as error:
        print(f"CUDA unavailable; using CPU. {type(error).__name__}: {error}", flush=True)
else:
    print("CPU comparison selected.", flush=True)
setup_seconds = perf_counter() - setup_started
gpu_upload_s = gpu_compute_s = gpu_download_s = 0.0
cpu_triangulation_s = cpu_assembly_s = 0.0
gpu_mesh_regions = gpu_normal_triangles = 0

# --- Reuse One Existing Layout or Upload ---
if not hasattr(shapely, "coverage_clean"):
    raise RuntimeError(
        "Restart the Colab session after installation, then rerun. "
        "Uploaded files stay available."
    )
if not 0 < THICKNESS_UM < float("inf"):
    raise ValueError("THICKNESS_UM must be finite and positive.")
if not 0 <= XY_TOLERANCE_UM < float("inf"):
    raise ValueError("XY_TOLERANCE_UM must be finite and nonnegative.")
if OUTPUT_FORMAT not in {"stl", "obj"}:
    raise ValueError("OUTPUT_FORMAT must be 'stl' or 'obj'.")

extensions = {".gds", ".gdsii", ".oas", ".oasis", ".dxf"}
existing = [p for p in Path.cwd().iterdir() if p.suffix.lower() in extensions]
if len(existing) == 1:
    source = existing[0]
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one GDS, OASIS, or DXF file.")
    source = Path(next(iter(uploaded)))

started = perf_counter()
print(f"Reading {source.name} ...", flush=True)
extension = source.suffix.lower()
polygons = []

# --- Read GDS or OASIS in Micrometres ---
if extension in {".gds", ".gdsii", ".oas", ".oasis"}:
    if extension in {".gds", ".gdsii"}:
        library = gdstk.read_gds(source, unit=1e-6)
    else:
        library = gdstk.read_oas(source, unit=1e-6)
    cells = {cell.name: cell for cell in library.top_level()}
    if not cells:
        raise ValueError("No top-level layout cell found.")
    cell_name = next(iter(cells))
    if len(cells) > 1:
        print("Top-level cells:", ", ".join(cells))
        cell_name = input("Enter the cell to extrude: ").strip()
    polygons = [shapely.Polygon(p.points) for p in cells[cell_name].get_polygons()]
    del cells, library

# --- Read DXF Outlines and Filled Regions ---
elif extension == ".dxf":
    document = ezdxf.readfile(source)
    unit_um = (
        units.conversion_factor(document.units, units.MM) * 1000
        if document.units else DXF_UNIT_UM
    )
    if not 0 < unit_um < float("inf"):
        raise ValueError("DXF_UNIT_UM must be finite and positive.")
    print(f"DXF scale: 1 drawing unit = {unit_um:g} um", flush=True)
    linework = {}
    entities = document.modelspace().query(
        "INSERT LINE LWPOLYLINE POLYLINE CIRCLE ARC ELLIPSE SPLINE "
        "HATCH MPOLYGON SOLID TRACE 3DFACE"
    )
    for entity in geo.gfilter(recursive_decompose(entities)):
        proxy = geo.proxy(entity, distance=0.01 / unit_um, force_line_string=True)
        proxy.places = 12
        geometry = shapely.force_2d(shape(proxy))
        if geometry.geom_type in {"Polygon", "MultiPolygon"}:
            polygons.append(geometry)
        elif geometry.geom_type in {"LineString", "MultiLineString"}:
            linework.setdefault(entity.dxf.layer, []).append(geometry)
    for curves in linework.values():
        polygons.append(shapely.build_area(shapely.union_all(curves)))
    polygons = [
        scale(p, xfact=unit_um, yfact=unit_um, origin=(0, 0)) for p in polygons
    ]
else:
    raise ValueError("Upload a GDS, OASIS, or DXF file.")

# --- Repair and Merge Without Rounding or Closing Gaps ---
polygons = np.asarray(polygons, dtype=object)
invalid = ~shapely.is_valid(polygons)
polygons[invalid] = shapely.make_valid(
    polygons[invalid], method="structure", keep_collapsed=False
)
polygons = shapely.get_parts(polygons)
polygons = polygons[(shapely.get_type_id(polygons) == 3) & ~shapely.is_empty(polygons)]
if not len(polygons):
    raise ValueError("No filled regions found. Join DXF edges into closed loops.")
print(
    f"[{perf_counter() - started:.1f} s] Merging {len(polygons):,} polygons ...",
    flush=True,
)
polygons = shapely.coverage_clean(polygons, gap_width=0.0, snapping_distance=0.0)
print(
    f"[{perf_counter() - started:.1f} s] Shared boundaries matched; combining regions ...",
    flush=True,
)
outline = shapely.coverage_union_all(polygons)
original_parts = shapely.get_parts(outline)
original_holes = int(shapely.get_num_interior_rings(original_parts).sum())
original_vertices = (
    int(shapely.get_num_coordinates(outline)) - original_holes - len(original_parts)
)
print(
    f"[{perf_counter() - started:.1f} s] Simplifying boundaries "
    f"with {XY_TOLERANCE_UM:g} um tolerance ...", flush=True,
)
original_rings = shapely.get_rings(original_parts)
polygon_offsets = np.r_[
    0, np.cumsum(1 + shapely.get_num_interior_rings(original_parts))
]
for attempt in range(5):
    used_tolerance_um = XY_TOLERANCE_UM / (2 ** attempt) if attempt < 4 else 0.0
    reduced_rings = shapely.simplify(
        original_rings, used_tolerance_um, preserve_topology=True
    )
    ring_sizes = shapely.get_num_coordinates(reduced_rings)
    if np.any(ring_sizes < 4):
        continue
    ring_offsets = np.r_[0, np.cumsum(ring_sizes)]
    candidate_parts = shapely.from_ragged_array(
        shapely.GeometryType.POLYGON,
        shapely.get_coordinates(reduced_rings),
        (ring_offsets, polygon_offsets),
    )
    candidate = shapely.multipolygons(candidate_parts)
    if shapely.is_valid(candidate):
        outline = candidate
        break
else:
    raise ValueError("The simplified footprint is invalid; inspect the input boundaries.")
parts = shapely.orient_polygons(shapely.get_parts(outline))
parts = parts[(shapely.get_type_id(parts) == 3) & ~shapely.is_empty(parts)]
hole_count = int(shapely.get_num_interior_rings(parts).sum())
boundary_vertices = (
    int(shapely.get_num_coordinates(parts).sum()) - hole_count - len(parts)
)
if len(parts) != len(original_parts) or hole_count != original_holes:
    raise ValueError("Simplification changed the region or hole count; reduce the tolerance.")
print(
    f"Boundary vertices: {original_vertices:,} -> {boundary_vertices:,}; "
    f"holes preserved: {hole_count:,}; XY tolerance used: {used_tolerance_um:g} um",
    flush=True,
)
del polygons, original_parts, original_rings, reduced_rings, candidate_parts, candidate
cpu_geometry_s = perf_counter() - started
print(
    f"[{perf_counter() - started:.1f} s] Triangulating {len(parts):,} regions ...",
    flush=True,
)

# --- Prepare All Holes Together and Extrude Each Region ---
vertex_blocks, face_blocks = [], []
offset = 0
for index, polygon in enumerate(parts):
    triangulation_started = perf_counter()
    rings = shapely.get_rings(polygon)
    counts = shapely.get_num_coordinates(rings)
    points = shapely.get_coordinates(rings)
    keep = np.ones(len(points), dtype=bool)
    keep[np.cumsum(counts) - 1] = False
    vertices = points[keep]
    ends = np.cumsum(counts - 1)
    edges = np.column_stack((np.arange(len(vertices)), np.arange(1, len(vertices) + 1)))
    edges[ends - 1, 1] = np.r_[0, ends[:-1]]
    holes = shapely.get_coordinates(
        shapely.point_on_surface(shapely.polygons(rings[1:]))
    )
    data = {"vertices": vertices, "segments": edges}
    if len(holes):
        data["holes"] = holes
    cap = triangle.triangulate(data, "pQY")
    if len(cap["vertices"]) != len(vertices):
        raise ValueError("Boundary triangulation changed; inspect this layout region.")
    cpu_triangulation_s += perf_counter() - triangulation_started
    n = len(vertices)
    use_device = gpu_available and n >= 100_000
    if use_device:
        estimated_bytes = 512 * n + 64 * len(cap["triangles"])
        use_device = estimated_bytes < cp.cuda.runtime.memGetInfo()[0] * 0.6
    xp = cp if use_device else np
    step_started = perf_counter()
    v = xp.asarray(vertices)
    f = xp.asarray(cap["triangles"], dtype=xp.int64)
    e = xp.asarray(edges, dtype=xp.int64)
    heights = xp.asarray([0.0, THICKNESS_UM])
    if use_device:
        cp.cuda.get_current_stream().synchronize()
        gpu_upload_s += perf_counter() - step_started
        step_started = perf_counter()
    extruded_vertices = xp.column_stack((
        xp.tile(v, (2, 1)), xp.repeat(heights, n)
    )) * 1e-3
    extruded_faces = xp.vstack((
        f[:, ::-1], f + n,
        xp.column_stack((e[:, 0], e[:, 1], e[:, 1] + n)),
        xp.column_stack((e[:, 0], e[:, 1] + n, e[:, 0] + n)),
    )) + offset
    if use_device:
        cp.cuda.get_current_stream().synchronize()
        gpu_compute_s += perf_counter() - step_started
        step_started = perf_counter()
        vertex_blocks.append(cp.asnumpy(extruded_vertices))
        face_blocks.append(cp.asnumpy(extruded_faces))
        gpu_download_s += perf_counter() - step_started
        gpu_mesh_regions += 1
    else:
        vertex_blocks.append(extruded_vertices)
        face_blocks.append(extruded_faces)
        cpu_assembly_s += perf_counter() - step_started
    del v, f, e, heights, extruded_vertices, extruded_faces
    offset += 2 * n
    if index == 0 or (index + 1) % 100 == 0 or index + 1 == len(parts):
        print(
            f"[{perf_counter() - started:.1f} s] Extruded {index + 1:,}/{len(parts):,}",
            flush=True,
        )

mesh = trimesh.Trimesh(
    vertices=np.vstack(vertex_blocks), faces=np.vstack(face_blocks), process=False
)
mesh.units = "mm"
print(
    f"Triangles: {len(mesh.faces):,}; "
    f"binary STL size: {(84 + 50 * len(mesh.faces)) / 1e6:.1f} MB",
    flush=True,
)
del vertex_blocks, face_blocks, data, cap, points, vertices
if gpu_available:
    cp.get_default_memory_pool().free_all_blocks()
if len(mesh.faces) <= 200_000 and not mesh.is_watertight:
    raise ValueError("The mesh is not watertight. Inspect touching or invalid edges.")
print(
    f"[{perf_counter() - started:.1f} s] Writing {len(mesh.faces):,} triangles ...",
    flush=True,
)

# --- Stream the Export to Keep Memory Use Bounded ---
output_path = source.with_name(source.stem + "_3D_mm." + OUTPUT_FORMAT)
export_started = perf_counter()
if OUTPUT_FORMAT == "stl":
    record_type = np.dtype([
        ("normal", "<f4", (3,)), ("vertices", "<f4", (3, 3)), ("attribute", "<u2")
    ])
    gpu_export = gpu_available
    export_vertices, export_faces = mesh.vertices, mesh.faces
    if gpu_export:
        needed = mesh.vertices.nbytes + mesh.faces.nbytes + 200_000_000
        gpu_export = needed < cp.cuda.runtime.memGetInfo()[0] * 0.6
    if gpu_export:
        step_started = perf_counter()
        export_vertices = cp.asarray(mesh.vertices)
        export_faces = cp.asarray(mesh.faces)
        cp.cuda.get_current_stream().synchronize()
        gpu_upload_s += perf_counter() - step_started
    xp = cp if gpu_export else np
    print(f"STL normals: {'GPU' if gpu_export else 'CPU'}", flush=True)
    with output_path.open("wb") as stream:
        stream.write(b"Layout extrusion; coordinates in millimetres".ljust(80, b" "))
        stream.write(struct.pack("<I", len(mesh.faces)))
        for first in range(0, len(mesh.faces), 100_000):
            step_started = perf_counter()
            triangles = export_vertices[export_faces[first:first + 100_000]]
            normals = xp.cross(
                triangles[:, 1] - triangles[:, 0], triangles[:, 2] - triangles[:, 0]
            )
            lengths = xp.linalg.norm(normals, axis=1, keepdims=True)
            normals /= xp.where(lengths > 0, lengths, 1.0)
            packed = xp.concatenate((normals[:, None, :], triangles), axis=1)
            packed = packed.astype(xp.float32)
            if gpu_export:
                cp.cuda.get_current_stream().synchronize()
                gpu_compute_s += perf_counter() - step_started
                step_started = perf_counter()
                packed = cp.asnumpy(packed)
                gpu_download_s += perf_counter() - step_started
                gpu_normal_triangles += len(packed)
            records = np.zeros(len(triangles), dtype=record_type)
            records["normal"], records["vertices"] = packed[:, 0], packed[:, 1:]
            records.tofile(stream)
            if first % 1_000_000 == 0:
                print(f"Exported {min(first + 100_000, len(mesh.faces)):,} triangles", flush=True)
    del export_vertices, export_faces, triangles, normals, lengths, packed, records
    if gpu_available:
        cp.get_default_memory_pool().free_all_blocks()
else:
    with output_path.open("w") as stream:
        stream.write("# Coordinates in millimetres\n")
        np.savetxt(stream, mesh.vertices, fmt="v %.12g %.12g %.12g")
        for first in range(0, len(mesh.faces), 100_000):
            np.savetxt(stream, mesh.faces[first:first + 100_000] + 1, fmt="f %d %d %d")

export_s = perf_counter() - export_started

print(
    f"[{perf_counter() - started:.1f} s] Finished: {output_path.name} "
    f"({output_path.stat().st_size / 1e6:.1f} MB)", flush=True,
)
print("Dimensions [X, Y, Z] in mm:", mesh.extents)
print(f"CPU geometry: {cpu_geometry_s:.3f} s; triangulation: {cpu_triangulation_s:.3f} s")
print(f"CPU mesh assembly: {cpu_assembly_s:.3f} s; export wall time: {export_s:.3f} s")
print(
    f"GPU upload: {gpu_upload_s:.3f} s; compute: {gpu_compute_s:.3f} s; "
    f"download: {gpu_download_s:.3f} s"
)
print(
    f"GPU work: {gpu_mesh_regions:,} mesh regions; "
    f"normals for {gpu_normal_triangles:,} triangles"
)
print(f"CUDA setup: {setup_seconds:.3f} s; conversion: {perf_counter() - started:.3f} s")
files.download(str(output_path))
if len(mesh.faces) <= 200_000:
    display(scene_to_notebook(mesh.scene()))
else:
    print("Large mesh: open the exported file in a desktop mesh viewer.")

GPU enabled: NVIDIA L4


Saving 400_Qubits.oas to 400_Qubits.oas
Reading 400_Qubits.oas ...
[352.0 s] Merging 8,109 polygons ...
[790.2 s] Shared boundaries matched; combining regions ...
[1145.7 s] Simplifying boundaries with 0.1 um tolerance ...
Boundary vertices: 6,210,617 -> 2,007,989; holes preserved: 311,452; XY tolerance used: 0.1 um
[1336.7 s] Triangulating 2,208 regions ...
[1336.7 s] Extruded 1/2,208
[1341.8 s] Extruded 100/2,208
[1341.8 s] Extruded 200/2,208
[1341.9 s] Extruded 300/2,208
[1341.9 s] Extruded 400/2,208
[1342.0 s] Extruded 500/2,208
[1342.1 s] Extruded 600/2,208
[1342.1 s] Extruded 700/2,208
[1342.2 s] Extruded 800/2,208
[1342.2 s] Extruded 900/2,208
[1342.3 s] Extruded 1,000/2,208
[1342.4 s] Extruded 1,100/2,208
[1350.2 s] Extruded 1,200/2,208
[1350.2 s] Extruded 1,300/2,208
[1350.3 s] Extruded 1,400/2,208
[1350.4 s] Extruded 1,500/2,208
[1350.4 s] Extruded 1,600/2,208
[1350.5 s] Extruded 1,700/2,208
[1350.5 s] Extruded 1,800/2,208
[1350.6 s] Extruded 1,900/2,208
[1350.7 s] Extruded 2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Large mesh: open the exported file in a desktop mesh viewer.
